# GO:BP / MSigDB GPU ORA: full models, ARCHS4-gene-set universe

**Environment:** `gpu-kmeans` (Python kernel, cupy/RAPIDS)

ARCHS4, GTEx, and Recount2 have different gene sets after QC filtering (18423 / 21613 /
6000 genes respectively), so running GPU ORA independently per dataset (each against its
own full universe) tests a different number of pathways per dataset, which makes the
recovered-pathway percentages across datasets not directly comparable.

Fix: use **ARCHS4's own gene set as the reference universe** for all three datasets,
rather than the 3-way gene intersection (which shrank everyone's universe down to
Recount2's much smaller gene panel and tanked ARCHS4's own recovered-pathway %, since
ARCHS4's own universe no longer matched its earlier non-shared-universe results).
ARCHS4 is evaluated against its own full gene set unchanged (identical universe size /
results to the original per-dataset analyses in `08_saturation_study/gpu_ora_bp` and
`gpu_ora_msigdb`); GTEx and Recount2 are each evaluated against ARCHS4's gene set
intersected with their own measured genes (you can't test a pathway against a gene a
dataset never measured).

This notebook does not modify `libs/gpu_ora.py` -- it calls its existing low-level
building blocks (`build_pathway_matrix`, `build_hit_matrix`, `hypergeom_ora`,
`bh_adjust_rows`, `combine_min_across_lvs`), which already accept an explicit
`universe_genes` list, directly.

In [1]:
import os
import time

import pandas as pd

REPO_ROOT = "/home/msubirana/Documents/pivlab/clamp-analyses"
os.chdir(REPO_ROOT)

import sys
sys.path.insert(0, os.path.join(REPO_ROOT, "libs"))
import gpu_ora

t_start = time.time()

## Model paths

In [2]:
archs4_seed_dirs = {
    seed: f"output/01_model_building/04_archs4/06_bp_coverage_rshall/06_bp_coverage_hall_rs_100/hall_coverage_rs100_seed_{seed}"
    for seed in [1, 2, 3]
}

models = {
    "ARCHS4": {
        "CLAMPfull": {seed: os.path.join(d, "CLAMPfull_hall", "Z.csv") for seed, d in archs4_seed_dirs.items()},
        "CLAMPbase": {seed: os.path.join(d, "CLAMPbase", "Z.csv") for seed, d in archs4_seed_dirs.items()},
    },
    "GTEx": {
        "CLAMPfull": {None: "output/01_model_building/02_gtex/10_CLAMP_hall/CLAMPfull_hall/Z.csv"},
        "CLAMPbase": {None: "output/01_model_building/02_gtex/01_CLAMP/CLAMPbase/Z.csv"},
    },
    "Recount2": {
        "CLAMPfull": {None: "output/01_model_building/03_recount2/01_recount2_hall/CLAMPfull_hall/Z.csv"},
        "CLAMPbase": {None: "output/01_model_building/03_recount2/00_recount2/CLAMPbase/Z.csv"},
    },
}

libraries = {
    "bp": "data/pathways/go_bp.Hs.symbols.gmt",
    "msigdb": "data/pathways/msigdb.v2026.1.Hs.symbols.gmt",
}

output_dir = "output/03_model_biology/00_archs4/08_saturation_study/gpu_ora_shared_universe"
os.makedirs(output_dir, exist_ok=True)

## Reference universe: ARCHS4's own gene set

CLAMPfull and CLAMPbase share the same gene index within a dataset (same input `Y`), so
this only needs reading one `Z.csv` per dataset.

In [3]:
def read_gene_index(z_path):
    return list(pd.read_csv(z_path, index_col=0, usecols=[0]).index)

archs4_genes = read_gene_index(models["ARCHS4"]["CLAMPfull"][1])
gtex_genes = set(read_gene_index(models["GTEx"]["CLAMPfull"][None]))
recount2_genes = set(read_gene_index(models["Recount2"]["CLAMPfull"][None]))

# evaluation universe per dataset = ARCHS4's gene set intersected with that dataset's own
# measured genes -- for ARCHS4 itself this is a no-op (identical to its own full universe,
# so its results match the original non-shared-universe analyses exactly)
dataset_universe = {
    "ARCHS4": archs4_genes,
    "GTEx": [g for g in archs4_genes if g in gtex_genes],
    "Recount2": [g for g in archs4_genes if g in recount2_genes],
}

for name, genes in dataset_universe.items():
    print(f"{name} evaluation universe: {len(genes)} genes")

ARCHS4 evaluation universe: 18423 genes
GTEx evaluation universe: 15289 genes
Recount2 evaluation universe: 5974 genes


## Run GPU ORA for every (library, dataset, model_type[, seed]) combo

In [4]:
def run_gpu_ora_with_universe(z_path, library, universe_genes, min_size=10, max_size=50000, pct=0.01):
    """Same end-to-end steps as gpu_ora.run_gpu_ora_for_model, but against an externally
    supplied universe_genes list instead of the model's own full gene index."""
    Z = pd.read_csv(z_path, index_col=0)
    N = len(universe_genes)

    P, term_names = gpu_ora.build_pathway_matrix(universe_genes, library, min_size=min_size, max_size=max_size)
    n_total_pathways = len(term_names)

    H, n_top = gpu_ora.build_hit_matrix(Z, universe_genes, pct=pct)

    pvals = gpu_ora.hypergeom_ora(H, P, N, n_top)
    padj_per_lv = gpu_ora.bh_adjust_rows(pvals)
    terms_padj = gpu_ora.combine_min_across_lvs(padj_per_lv, term_names)

    b_path = "/".join(str(z_path).split("/")[:-1]) + "/B.csv"
    try:
        with open(b_path) as f:
            header = f.readline()
        n_samples = header.rstrip("\n").count(",")
    except FileNotFoundError:
        n_samples = None

    return {
        "n_samples": n_samples,
        "n_lvs": Z.shape[1],
        "n_top_genes": n_top,
        "n_total_msigdb": n_total_pathways,
        "terms_padj": terms_padj,
    }


summary_rows = []

for lib_name, lib_path in libraries.items():
    print(f"=== Loading library: {lib_name} ===")
    library = gpu_ora.read_gmt(lib_path)
    print(f"{lib_name}: {len(library)} gene sets loaded")
    lib_dir = os.path.join(output_dir, lib_name)
    os.makedirs(lib_dir, exist_ok=True)

    for dataset, model_types in models.items():
        universe_genes = dataset_universe[dataset]

        for model_type, seed_paths in model_types.items():
            for seed, z_path in seed_paths.items():
                if not os.path.exists(z_path):
                    print(f"SKIP (no Z.csv): {lib_name} {dataset} {model_type} seed={seed} -> {z_path}")
                    continue

                tag = f"{dataset}_{model_type}" + (f"_seed{seed}" if seed is not None else "")
                cache_path = os.path.join(lib_dir, f"{tag}_gpu_ora.csv")
                meta_path = os.path.join(lib_dir, f"{tag}_meta.csv")

                if os.path.exists(cache_path) and os.path.exists(meta_path):
                    print(f"Loading cached: {lib_name} {tag}")
                    meta = pd.read_csv(meta_path).iloc[0]
                else:
                    print(f"Running GPU ORA: {lib_name} {tag}")
                    t0 = time.time()
                    res = run_gpu_ora_with_universe(
                        z_path, library, universe_genes, min_size=10, max_size=50000, pct=0.01,
                    )
                    print(f"  done in {time.time()-t0:.2f}s")
                    res["terms_padj"].rename_axis("term").reset_index(name="padj_min_across_lvs").to_csv(cache_path, index=False)
                    meta = pd.Series({k: v for k, v in res.items() if k != "terms_padj"})
                    meta.to_frame().T.to_csv(meta_path, index=False)

                summary_rows.append({
                    "library": lib_name, "dataset": dataset, "model_type": model_type, "seed": seed,
                    "n_samples": meta["n_samples"], "n_lvs": meta["n_lvs"],
                    "n_top_genes": meta["n_top_genes"], "n_total_pathways": meta["n_total_msigdb"],
                })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(output_dir, "results_shared_universe_summary.csv"), index=False)
print(f"Collected {len(summary_df)} rows")
print(summary_df)

=== Loading library: bp ===
bp: 15413 gene sets loaded
Running GPU ORA: bp ARCHS4_CLAMPfull_seed1


  done in 3.39s
Running GPU ORA: bp ARCHS4_CLAMPfull_seed2


  done in 2.54s
Running GPU ORA: bp ARCHS4_CLAMPfull_seed3


  done in 2.42s
Running GPU ORA: bp ARCHS4_CLAMPbase_seed1


  done in 2.38s
Running GPU ORA: bp ARCHS4_CLAMPbase_seed2


...

  done in 5.86s
Running GPU ORA: msigdb GTEx_CLAMPfull


  done in 3.94s
Running GPU ORA: msigdb GTEx_CLAMPbase


  done in 3.97s
Running GPU ORA: msigdb Recount2_CLAMPfull


  done in 2.30s
Running GPU ORA: msigdb Recount2_CLAMPbase


  done in 1.62s
Collected 20 rows
   library   dataset model_type  seed  n_samples  n_lvs  n_top_genes  \
0       bp    ARCHS4  CLAMPfull   1.0     605614   1728          185   
1       bp    ARCHS4  CLAMPfull   2.0     605614   1728          185   
2       bp    ARCHS4  CLAMPfull   3.0     605614   1728          185   
3       bp    ARCHS4  CLAMPbase   1.0     605614   1728          185   
4       bp    ARCHS4  CLAMPbase   2.0     605614   1728          185   
5       bp    ARCHS4  CLAMPbase   3.0     605614   1728          185   
6       bp      GTEx  CLAMPfull   NaN      17382    578          153   
7       bp      GTEx  CLAMPbase   NaN      17382    578          153   
8       bp  Recount2  CLAMPfull   NaN      37032    724           60   
9       bp  Recount2  CLAMPbase   NaN      37032    724           60   
10  msigdb    ARCHS4  CLAMPfull   1.0     605614   1728          185   
11  msigdb    ARCHS4  CLAMPfull   2.0     605614   1728          185   
12  msigdb    ARCHS4  CLAMPful

In [5]:
print(f"Total notebook time: {(time.time()-t_start)/60:.1f} min")

Total notebook time: 1.1 min
